# An Interactive Workshop on Synthetic Mobility Data Generation

**A hands-on afternoon: make one, test one, and decide what may be published** · CASA, UCL · 13:00–17:00

---

## The participant workbook

This notebook is the part of the afternoon that is **in your hands**. Everything else —
the real days, the aggregates, the reveal — happens on the big screen, together.

You will open it **three times**:

| when | what you do here |
|---|---|
| **13:20** | **② Draw a day** — make one up, out of your head, before we show you anything |
| **15:00** | **③ Real or synthetic?** — ten pairs; one of each pair is a real person |
| **15:30** | **④ Try the methods** — run the approaches from the talk and compare what they make |

Each step ends with a short code that you paste into a form. That is how your answer
reaches the screen at the front.

**You do not need to read or write any code.** Every step below is a panel: click things,
press the ▶ button on the left when asked, look at what comes out.

**No Google account, or would rather not sign in?** Tell one of the helpers and we will
pair you with a neighbour.

---

**Run this first.** It takes about twenty seconds.

In [ ]:
#@title ① Get ready { display-mode: "form" }
#@markdown Press ▶ on the left. You only ever need to do this once.

#@markdown Leave this blank unless the facilitator reads out an address. You can paste a
#@markdown link to the bundle zip here, or the path of one you uploaded on the left.
BUNDLE_URL = ""  #@param {type:"string"}

import base64, io, json, os, shutil, sys, urllib.error, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.lines import Line2D

# Where the data bundle is downloaded from. The second address is a mirror, tried if the first fails.
BUNDLE_URLS = [
    # always the newest release, so publishing the event data changes nothing here
    "https://github.com/PangYanbo/synthetic-mobility-workshop/releases/latest/download/participant_bundle.zip",
    # add a mirror here (e.g. a Google Drive direct-download link) before the day
]
BUNDLE = Path("bundle")


def _local_candidates():
    """Look for a bundle already sitting next to the notebook (a folder or a zip)."""
    seen, out = set(), []
    for base in [Path.cwd(), *list(Path.cwd().parents)[:5], Path("/content")]:
        for rel in ("mock_bundle", "apps/public-lab/mock_bundle",
                    "mock_bundle.zip", "apps/public-lab/mock_bundle.zip",
                    "bundle", "bundle.zip"):
            c = base / rel
            if c.exists() and str(c) not in seen:
                seen.add(str(c)); out.append(c)
    return out


def _unpack(src: Path, dest: Path) -> None:
    if dest.exists():
        shutil.rmtree(dest)
    if src.is_dir():
        shutil.copytree(src, dest)
    else:
        with zipfile.ZipFile(src) as z:
            z.extractall(dest)


def _fetch(targets):
    """Try each address in turn; the first that downloads wins. Returns (blob, target, err)."""
    blob, target, err = None, (targets[-1] if targets else ""), None
    for t in targets:
        try:
            with urllib.request.urlopen(t, timeout=120) as r:
                blob = r.read()
            target = t
            break
        except (urllib.error.URLError, ValueError, OSError) as exc:
            err = exc
    return blob, target, err


def _load_bundle(url: str) -> str:
    """Order: a local path you typed, then an address you typed, then a bundle already next to
    the notebook, then the default addresses. An address you type always wins over a stray local copy.
    """
    if url:
        p = Path(url)
        if p.exists():
            _unpack(p, BUNDLE)
            return f"local path  {p}"
        blob, target, err = _fetch([url])
        if blob is not None:
            with zipfile.ZipFile(io.BytesIO(blob)) as z:
                z.extractall(BUNDLE)
            return f"downloaded  {target}"
        print(f"  (could not fetch {url} — {err}; looking for a bundle next to the notebook)")
    for c in _local_candidates():
        if c.resolve() == BUNDLE.resolve():
            continue
        _unpack(c, BUNDLE)
        return f"local bundle  {c}"
    blob, target, err = _fetch([url] if url else BUNDLE_URLS)
    if blob is None:
        e = err
        raise SystemExit(
            "\nCould not load the data bundle.\n"
            f"  tried: {target}\n  reason: {e}\n\n"
            "  · On the day: put your hand up, someone will come over.\n"
            "  · Debugging locally: run\n"
            "        python apps/public-lab/build_mock_bundle.py\n"
            "    and re-run this cell from anywhere inside the repository.\n"
            "  · Or upload the bundle zip into this session (files panel on the left)\n"
            "    and press play again — it is found automatically."
        ) from None
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(BUNDLE)
    return f"downloaded  {target}"


_source = _load_bundle(BUNDLE_URL)

META       = json.loads((BUNDLE / "meta.json").read_text())
GROUPS     = json.loads((BUNDLE / "groups.json").read_text())
ACTIVITIES = json.loads((BUNDLE / "activities.json").read_text())
OUTLINE    = json.loads((BUNDLE / "london_outline.json").read_text())
PANEL      = pd.read_parquet(BUNDLE / "panel_synthetic.parquet")
CELLS_H3   = pd.read_parquet(BUNDLE / "h3_cells.parquet").set_index("h3")
# Hexagon outlines are only used on the big screen; the participant bundle has cell centres only.
_rings = BUNDLE / "h3_rings.parquet"
RINGS = pd.read_parquet(_rings).set_index("h3") if _rings.exists() else None

GROUP_BY_NAME = {g["display"]: g for g in GROUPS}
COLOR = {k: v["color"] for k, v in ACTIVITIES.items()}
LETTER_OF = {k: v.get("letter", k[0]) for k, v in ACTIVITIES.items()}
ACT_OF = {v: k for k, v in LETTER_OF.items()}
LABEL = {k: v["display"] for k, v in ACTIVITIES.items()}

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.spines.top": False,
    "axes.spines.right": False, "axes.spines.left": False,
})


def _slot_label(s: int) -> str:
    return f"{s // 2:02d}:{'30' if s % 2 else '00'}"


def draw_day_strip(acts, ax, title=""):
    """One day, midnight to midnight, as 48 half-hour slots.

    Drawn as runs rather than 48 separate boxes: what matters visually is how
    long each thing lasted, not that the clock ticks every half hour.
    """
    s = 0
    while s < len(acts):
        e = s
        while e + 1 < len(acts) and acts[e + 1] == acts[s]:
            e += 1
        blank = acts[s] is None
        ax.add_patch(plt.Rectangle((s, 0), e - s + 1, 1,
                                   facecolor="none" if blank else COLOR.get(acts[s], "#cccccc"),
                                   hatch="///" if blank else None,
                                   edgecolor="#d8d8d8" if blank else "white",
                                   linewidth=0.9))
        s = e + 1
    ax.set_xlim(0, 48); ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks(range(0, 49, 6))
    ax.set_xticklabels([_slot_label(s) if s < 48 else "24:00" for s in range(0, 49, 6)])
    ax.set_title(title, loc="left", fontsize=9)
    for sp in ax.spines.values():
        sp.set_visible(False)


def draw_day_map(locs, ax, title="", as_cells=False):
    """Where the day happened, on a London outline.

    At London scale a place is drawn as a single dot, sized by how long the
    person stayed. The underlying location is an area (a neighbourhood in the
    real data), and there is no finer view available — that is a deliberate
    limit, and one of the things this room will be deciding about this afternoon.

    `as_cells=True` draws the actual hexagons instead, if the ring geometry is
    present. That file only ships to the facilitators, for zoomed-in panels on
    the big screen — the hexagons are invisible at London scale anyway.
    """
    for poly in OUTLINE["polygons"]:
        xs = [p[0] for p in poly]; ys = [p[1] for p in poly]
        ax.plot(xs, ys, color="#bbbbbb", linewidth=0.8, zorder=1)

    seq, dwell = [], {}
    for a_loc in locs:
        if a_loc is None or (isinstance(a_loc, float) and np.isnan(a_loc)):
            continue
        dwell[a_loc] = dwell.get(a_loc, 0) + 1
        if not seq or seq[-1] != a_loc:
            seq.append(a_loc)

    pts = [(CELLS_H3.loc[h].cx, CELLS_H3.loc[h].cy) for h in seq if h in CELLS_H3.index]
    if len(pts) > 1:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], color="#7a3d00",
                linewidth=1.0, alpha=0.55, zorder=2)

    for h, n in dwell.items():
        if h not in CELLS_H3.index:
            continue
        row = CELLS_H3.loc[h]
        if as_cells and RINGS is not None and h in RINGS.index:
            ring = RINGS.loc[h]
            ax.add_patch(MplPolygon(list(zip(ring.ring_lon, ring.ring_lat)),
                                    closed=True, facecolor="#F58518",
                                    alpha=min(0.3 + n / 48 * 0.7, 1.0),
                                    edgecolor="#7a3d00", linewidth=0.6, zorder=3))
        else:
            ax.plot(row.cx, row.cy, "o", markersize=4 + 13 * (n / 48) ** 0.6,
                    color="#F58518", markeredgecolor="#7a3d00",
                    markeredgewidth=0.7, alpha=0.9, zorder=3)

    ax.set_aspect(1 / np.cos(np.deg2rad(51.5)))
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, loc="left", fontsize=9)
    for sp in ax.spines.values():
        sp.set_visible(False)


def show_day(acts, locs, title="", legend=True):
    fig = plt.figure(figsize=(11, 3.1))
    gs = fig.add_gridspec(1, 2, width_ratios=[2.6, 1], wspace=0.12)
    ax0 = fig.add_subplot(gs[0]); ax1 = fig.add_subplot(gs[1])
    draw_day_strip(acts, ax0, title)
    draw_day_map(locs, ax1, "where  ·  dot size = time spent there")
    if legend:
        present = [a for a in ACTIVITIES if a in set(acts)]   # gaps are not listed in the legend
        ax0.legend(handles=[Line2D([], [], marker="s", linestyle="", markersize=7,
                                   color=COLOR[a], label=LABEL[a]) for a in present],
                   loc="upper left", bbox_to_anchor=(0, -0.35), ncol=4,
                   frameon=False, fontsize=8)
    plt.show()


# ── Shared map projection and sampling helpers ───────────────────────────
_OUTLINE = json.loads((BUNDLE / "london_outline.json").read_text())["polygons"]
_xs = [p[0] for poly in _OUTLINE for p in poly]
_ys = [p[1] for poly in _OUTLINE for p in poly]
_X0, _X1, _Y0, _Y1 = min(_xs), max(_xs), min(_ys), max(_ys)
_K = np.cos(np.deg2rad((_Y0 + _Y1) / 2))
_SC = min(1000 / ((_X1 - _X0) * _K), 700 / (_Y1 - _Y0)) * 0.94
_OX = (1000 - (_X1 - _X0) * _K * _SC) / 2
_OY = (700 - (_Y1 - _Y0) * _SC) / 2
PX = lambda x: round(_OX + (x - _X0) * _K * _SC, 1)
PY = lambda y: round(700 - _OY - (y - _Y0) * _SC, 1)
BASE_PATH = "".join("M" + "L".join(f"{PX(a)},{PY(b)}" for a, b in poly) + "Z"
                    for poly in _OUTLINE)

MSOA_NAME = {}
try:
    for _f in json.loads((BUNDLE / "msoa_london.geojson").read_text())["features"]:
        MSOA_NAME[_f["properties"]["code"]] = _f["properties"].get("name", "")
except Exception:
    pass


def day_geometry(h3_seq, act_seq, msoa_seq=None):
    """A day's slots -> the points and path for its map: one point per place, activity shares for the wedges."""
    msoa_seq = msoa_seq if msoa_seq is not None else [None] * len(h3_seq)
    dwell, label, mix = {}, {}, {}
    for h, a, m in zip(h3_seq, act_seq, msoa_seq):
        if not isinstance(h, str):
            continue
        dwell[h] = dwell.get(h, 0) + 1
        if isinstance(m, str) and h not in label:
            label[h] = MSOA_NAME.get(m, "")
        if isinstance(a, str) and a != "travel":
            mix.setdefault(h, {})
            mix[h][a] = mix[h].get(a, 0) + 1
    seq, seen = [], None
    for h in h3_seq:
        if isinstance(h, str) and h != seen:
            seq.append(h); seen = h
    dots, done = [], []
    for h in seq:
        if h in CELLS_H3.index and h not in done:
            done.append(h)
            mm = mix.get(h) or {"other": 1}
            tot = sum(mm.values()) or 1
            shares = sorted(((LETTER_OF.get(a, "o"), round(c / tot, 3))
                             for a, c in mm.items()), key=lambda t: -t[1])
            dots.append([PX(CELLS_H3.loc[h].cx), PY(CELLS_H3.loc[h].cy),
                         round(dwell[h] / 48, 3), label.get(h, ""), shares])
    path = [[PX(CELLS_H3.loc[h].cx), PY(CELLS_H3.loc[h].cy)]
            for h in seq if h in CELLS_H3.index]
    return dots, path


def _offer_file(name, page, lead=""):
    """Save the panel as a page of its own, and say how to open it if the panel stays blank.

    In Colab the saved file lives on Google's machine, so a path to it is useless to you;
    there the fix is simply to run the step again. Elsewhere the file is on your own disk.
    """
    Path(name).write_text(page, encoding="utf-8")
    if "google.colab" in sys.modules:
        print((lead + " " if lead else "") + "If the panel above stays blank, press ▶ on this step again.")
        return
    from IPython.display import HTML as _H, display as _d
    _d(_H('<p style="font:13px sans-serif;color:#6b7280;margin:6px 0">'
          + (lead + " " if lead else "")
          + 'If the panel above is blank, <a href="' + name + '" target="_blank">open it in a new tab</a>'
          + ' or open this file in a browser: <code>' + str(Path(name).resolve()) + '</code></p>'))


_mock = str(META.get("bundle_version", "")).startswith("mock")
print("Ready.\n")
print(f"  source: {_source}")
print(f"  {META['n_synthetic_days']:,} synthetic days in the bundle")
print("  places shown as " + (META.get("place_unit_text")
      or f"H3 cells at resolution {META['h3_resolution']} (about 300 m across)"))
if _mock:
    print("\n" + "!" * 66)
    print("!! MOCK BUNDLE — every number and every day below is FAKE.")
    print("!! For rehearsal only. Do not show any of this on the day.")
    print("!" * 66)

---

`13:20` · **step 1 of 3**

## Draw a day

Before anyone tells you anything about what we do here — **make one up.**

Someone else. Not you. No names.

**How it works.** A day is drawn as **forty-eight half-hour slots**, midnight to midnight.
You pick an activity from the palette, then paint it onto the day — click one half-hour,
or drag across several. Every stretch you paint then gets a **place**: click the stretch,
then click an area on the map.

**Two things worth knowing before you start.**

- **You do not have to fill the whole day.** Leave gaps if that is what feels right.
- **There is no right answer.** We are genuinely interested in what a room full of people
  imagines an ordinary day to look like — that is the point of doing this before anything
  is explained.

When you are done you get a short code. **Paste it into the form** (the panel gives you a
link that already has it filled in), and watch your day appear on the screen at the front.

**We put these on the screen this afternoon and delete them at the end of the day.**

In [ ]:
#@title ② Draw a day { display-mode: "form" }
#@markdown Leave the box below alone unless the facilitator reads out an address.

FORM_PREFILL_URL = "https://docs.google.com/forms/d/e/1FAIpQLScvZHGRZ5I0g2cjlsLX6B-aaLNxHuUu74WZSKetFI8SzXJq3w/viewform?usp=pp_url&entry.1287034676="

from IPython.display import HTML, display

_acts = json.loads((BUNDLE / "activities.json").read_text())
_msoa = json.loads((BUNDLE / "msoa_london.geojson").read_text())
_page = (BUNDLE / "day_canvas.html").read_text()



def _receive(payload):
    """Colab hands the canvas result back here."""
    globals()["_IMAGINED"] = payload


_via = "web"
try:
    from google.colab import output as _colab_output
    _colab_output.register_callback("public_lab.submit_day", _receive)
    _via = "colab"
except Exception:
    pass

_page = (_page
    # Leaflet is inlined because Colab blocks scripts loaded from outside the output frame.
    .replace("/*__LEAFLET_CSS__*/", (BUNDLE / "leaflet.min.css").read_text())
    .replace("/*__LEAFLET_JS__*/", (BUNDLE / "leaflet.min.js").read_text())
    .replace("/*__ACTIVITIES__*/ {}", json.dumps(_acts))
    .replace('/*__MSOA__*/ {"type":"FeatureCollection","features":[]}',
             json.dumps(_msoa, separators=(",", ":")))
    .replace("/*__CONFIG__*/ {}",
             json.dumps({"via": _via, "form_url": FORM_PREFILL_URL})))

# Rendered inside an <iframe srcdoc>, the one approach that works in Colab, JupyterLab and VS Code alike.
import html as _html

# Wrapped in a <div> so IPython does not print a warning box under the panel.
display(HTML(
    '<div style="max-width:900px">'
    '<iframe srcdoc="' + _html.escape(_page, quote=True) + '" '
    'width="100%" height="1280" frameborder="0" '
    'style="border:0;display:block"></iframe></div>'))

_offer_file("draw_a_day.html", _page)

---

`15:00` · **step 2 of 3**

## Real or synthetic?

Ten pairs of days. In each pair, **one belongs to a real person and one was generated.**

Pick the one you think is real, or press **“I can't tell them apart”** — that is a real
answer, and it is counted separately on the screen rather than folded into the score.
**You will get no feedback here** — we do the reveal together on the screen, once everyone
has answered.

> The real days you are about to see have had their **observation gaps filled in**.
> That is not cosmetic: the generator is trained on completed days, so a completed day
> is what the model is actually imitating. Comparing against the raw, gap-riddled version
> would be comparing two different things — and we will come back to what those filled-in
> records mean when we talk about what may be published.

In [ ]:
#@title ③ Real or synthetic? { display-mode: "form" }
FORM_PREFILL_URL_PAIRS = "https://docs.google.com/forms/d/e/1FAIpQLSe2FgDhZcOLrpPXp8ZY18ZbOpm5wObaouq_h0EegwhKLbypnQ/viewform?usp=pp_url&entry.1943344602="

_jd = pd.read_parquet(BUNDLE / "judgment_days.parquet").sort_values(["pair_id", "side", "slot"])

_pairs = []
for pid, grp in _jd.groupby("pair_id"):
    entry = {"pair": int(pid)}
    for side, sg in grp.groupby("side"):
        sg = sg.sort_values("slot")
        letters = "".join(LETTER_OF.get(a, "_") if isinstance(a, str) else "_"
                          for a in sg.activity)
        dots, path = day_geometry(sg.h3.tolist(), sg.activity.tolist(),
                                  sg.msoa.tolist() if "msoa" in sg.columns else None)
        entry[side] = {"slots": letters, "places": [], "dots": dots, "path": path}
    if "A" in entry and "B" in entry:
        _pairs.append(entry)


def _receive_pairs(payload):
    globals()["_PAIRS"] = payload


try:
    from google.colab import output as _co2
    _co2.register_callback("public_lab.submit_pairs", _receive_pairs)
except Exception:
    pass

_page2 = ((BUNDLE / "pair_test.html").read_text()
    .replace("/*__PAIRS__*/ []", json.dumps(_pairs, separators=(",", ":")))
    .replace("/*__ACTIVITIES__*/ {}", json.dumps(_acts))
    .replace("/*__CONFIG__*/ {}",
             json.dumps({"base": BASE_PATH, "form_url": FORM_PREFILL_URL_PAIRS})))

display(HTML(
    '<div style="max-width:1000px">'
    '<iframe srcdoc="' + _html.escape(_page2, quote=True) + '" '
    'width="100%" height="900" frameborder="0" style="border:0;display:block"></iframe></div>'))

_offer_file("real_or_synthetic.html", _page2, lead=str(len(_pairs)) + " pairs.")

---

`15:30` · **step 3 of 3**

## Try the methods

You drew a day out of your head, and you have just tried to tell real from generated. Now
the machine's turn, using the approaches from the talk.

It takes only a few things from you — **which behavioural group**, **which kind of day**,
**where the day starts**, and **which way of generating**. Everything else, all forty-eight
half-hours of it, it decides.

The groups were **learned from the data, not assigned**. They are named by what the
behaviour looks like, never by a job — from movement alone you cannot tell a job from
anything else that looks like one.

**Press “Make one” twice without changing anything.** That is the whole point of this step.

*(The model itself lives on a server; this notebook draws from a library it already
generated. That library is a sample from the model, so it behaves exactly as it would live.)*

In [ ]:
#@title ④ Try the methods { display-mode: "form" }
from IPython.display import HTML, display
import html as _html

_groups = json.loads((BUNDLE / "groups.json").read_text())
_methods = (json.loads((BUNDLE / "methods.json").read_text())
            if (BUNDLE / "methods.json").exists()
            else [{"method_id": "", "display": "the model", "note": ""}])
_acts = json.loads((BUNDLE / "activities.json").read_text())

# A library of days for every combination of group, day type, method and start; each press draws from these
_N_PER = 200
_has_method = "method" in PANEL.columns
_P = PANEL.sort_values(["synth_id", "slot"])
_first = _P[_P.slot == 0].set_index("synth_id")
_made = {}
for sid, d in _P.groupby("synth_id", sort=False):
    f = _first.loc[sid]
    k = (f.group_id + "|" + f.day_type + "|" + (f.method if _has_method else "")
         + "|" + str(int(f.activity == "home")))
    lst = _made.setdefault(k, [])
    if len(lst) >= _N_PER:
        continue
    dots, path = day_geometry(d.h3.tolist(), d.activity.tolist())
    lst.append({"slots": "".join(LETTER_OF.get(a, "_") for a in d.activity),
                "dots": dots, "path": path})

_page3 = ((BUNDLE / "make_a_londoner.html").read_text()
    .replace("/*__MADE__*/ {}", json.dumps(_made, separators=(",", ":")))
    .replace("/*__GROUPS__*/ []", json.dumps(_groups))
    .replace("/*__METHODS__*/ []", json.dumps(_methods))
    .replace("/*__ACTIVITIES__*/ {}", json.dumps(_acts))
    .replace("/*__CONFIG__*/ {}", json.dumps({"base": BASE_PATH})))

display(HTML(
    '<div style="max-width:1020px">'
    '<iframe srcdoc="' + _html.escape(_page3, quote=True) + '" '
    'width="100%" height="1150" frameborder="0" style="border:0;display:block"></iframe></div>'))

_offer_file("try_the_methods.html", _page3, lead=str(len(_made)) + " settings available.")

---

You made a handful of choices. The machine made forty-eight — and it will make
forty-eight different ones next time you ask.